### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="body_density_prediction",
    dataset_year="1985",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="Kaggle", # Other OpenML. Original UCI source is gone.
    original_dataset_source_download_link="https://www.kaggle.com/datasets/fedesoriano/body-fat-prediction-dataset",
    download_description="""
We download the data from Kaggle.

kaggle datasets download fedesoriano/body-fat-prediction-dataset && unzip body-fat-prediction-dataset.zip && rm body-fat-prediction-dataset.zip
mkdir -p local-data-warehouse/body_density_prediction && mv bodyfat.csv local-data-warehouse/body_density_prediction/
""",
    # References
    academic_reference_bibtex=r"""@article{penrose1985generalized,
  title={Generalized body composition prediction equation for men using simple measurement techniques},
  author={Penrose, Keith W and Nelson, Arnold G and Fisher, Arnold Garth},
  journal={Medicine \& Science in Sports \& Exercise},
  volume={17},
  number={2},
  pages={189},
  year={1985},
  publisher={Ovid Technologies (Wolters Kluwer Health)}
}
""",
    academic_reference_bibtex_key="penrose1985generalized",
    license="None",
    data_tags=["IID"],
    curation_comments="""
We start with the data from Kaggle.

- Note, the task is not about bodyfat prediction as this is determined by a deterministic formula. Instead, the task is to estimate the density (which can be used to get the body fat via the deterministic formula). However, density requires a test. So we aim to predict from data the density such that we can skip this test. Which can make a lot of sense, if the test is too expensive as it requires underwater weighing.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Density",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "bodyfat.csv")
print("Loaded data shape:", df.shape)

# Bodyfat leaks the density
df = df.drop(columns=["BodyFat"])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (252, 15)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 252
Columns: 14
Use sampling: False (sample size: 252)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Weight', 'Abdomen', 'Chest', 'Hip', 'Thigh', 'Biceps', 'Neck', 'Knee', 'Forearm', 'Ankle']
Rows remaining as candidates after top-10 filter: 0 (of 252)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Density,Age,Weight,Height,Neck,Chest,Abdomen,Hip,Thigh,Knee,Ankle,Biceps,Forearm,Wrist
0,1.0549,35,217.00,73.75,40.5,107.5,95.1,104.5,64.8,41.3,25.6,36.4,33.7,19.4
1,1.0549,26,181.00,69.75,36.4,105.1,90.7,100.3,58.4,38.3,22.9,31.9,27.8,17.7
2,1.0355,43,183.25,70.00,37.1,108.0,105.0,103.0,63.7,40.0,23.6,33.5,27.8,17.4
3,1.0521,35,177.25,71.00,38.4,100.5,90.3,98.7,57.8,37.3,22.4,31.0,28.7,17.7
4,1.0607,40,158.00,69.25,36.3,97.0,86.6,92.6,55.9,36.3,22.1,29.8,26.3,17.3


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Density,float64,0.0,0.0,218.0,"1.061, 1.0414, 1.0484, 1.0524, 1.0549, 1.0603, 1.0678, 1.0373, 1.0403, 1.0575"
1,Weight,float64,0.0,0.0,197.0,"152.25, 172.75, 167.0, 168.25, 170.75, 179.75, 184.25, 177.25, 168.0, 161.75"
2,Height,float64,0.0,0.0,48.0,"71.5, 69.25, 69.5, 72.25, 67.5, 70.0, 69.75, 67.75, 73.5, 68.5"
3,Neck,float64,0.0,0.0,90.0,"38.5, 38.0, 37.4, 37.8, 36.5, 38.7, 35.5, 40.8, 40.2, 37.5"
4,Chest,float64,0.0,0.0,174.0,"99.1, 102.7, 94.0, 99.6, 98.9, 97.8, 89.2, 105.6, 101.8, 93.5"
5,Abdomen,float64,0.0,0.0,185.0,"88.7, 100.5, 89.7, 82.8, 95.0, 98.6, 92.4, 100.0, 83.6, 99.8"
6,Hip,float64,0.0,0.0,152.0,"98.3, 100.6, 99.3, 96.2, 102.5, 94.5, 101.7, 99.6, 101.6, 94.0"
7,Thigh,float64,0.0,0.0,139.0,"58.9, 54.7, 59.3, 58.5, 56.0, 57.5, 54.3, 59.1, 60.6, 60.0"
8,Knee,float64,0.0,0.0,90.0,"39.0, 37.3, 38.1, 38.0, 38.3, 38.7, 40.0, 39.4, 38.4, 36.2"
9,Ankle,float64,0.0,0.0,61.0,"22.6, 22.0, 22.5, 21.8, 22.7, 23.2, 22.4, 21.5, 23.4, 24.0"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Density,252.0,1.055574,0.019031,0.995,1.1089
Age,252.0,44.884921,12.602040,22.000,81.0000
Weight,252.0,178.924405,29.389160,118.500,363.1500
Height,252.0,70.148810,3.662856,29.500,77.7500
Neck,252.0,37.992063,2.430913,31.100,51.2000
Chest,252.0,100.824206,8.430476,79.300,136.2000
Abdomen,252.0,92.555952,10.783077,69.400,148.1000
Hip,252.0,99.904762,7.164058,85.000,147.7000
Thigh,252.0,59.405952,5.249952,47.200,87.3000
Knee,252.0,38.590476,2.411805,33.000,49.1000


In [7]:
# Categorical Feature Statistics
cat_stats

'No categorical/object features to summarize.'

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,-0.02,-0.066,0.0,0.0,log,7371.7,1.090559e+13,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to body_density_prediction/019d9cc4-a901-7b17-8ea9-277e92b37579
019d9cc4-a901-7b17-8ea9-277e92b37579
e30fa65b7db8c8ec6144329a82ac25fc5b6d6645388328f36b938792b7f88e41
